# Can GD reach ctx-count? — many initializations, frozen W\* readout

The architecture is the `DisentangledTransformer` with **`{W_v2, W_o2, fc_out}` frozen to
the W\* selection** (so the output is a convex combination of input token one-hots: ≥0,
sums to 1, NO softmax head).  W\* proves a parameter setting in this exact space reaches
**ctx-count**.  Question: does GD *find* it — from any initialization?

We train the attention (`W_q/W_k/W_v/W_o` layer 1, `W_q/W_k` layer 2) from several inits:
3 random seeds, a **small**-scale and a **large**-scale init (probes the attention-sharpening
barrier), and a **warm-start from W\*** (probes whether ctx-count is even a *stable* attractor).
Every converged curve is plotted against **ctx-count**, **ideal-α** (Bayes floor) and the
**W\* construction** (the optimal this frozen-readout architecture can represent).

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
DEVICE       = 'cuda:1'
DATA_DIR     = 'dataset'
MODELS_DIR   = 'models'
RESULTS_PATH = 'models/multi_init_results.json'   # cache (resumable)

ATTN_DIM   = 220          # = V + seq_len; matches the W* construction
SEQ_LEN    = 200
V, V_SIG   = 20, 10
T0, T1     = 100, 200     # CE evaluated on this window

THRESHOLDS = [0.1, 0.3, 0.5, 0.7, 0.9]   # representative sweep (set to all 9 for final)
INITS      = ['default-0', 'default-1', 'default-2', 'small', 'large', 'wstar']

EPOCHS       = 40         # enough to converge (loss plateaus well before this)
TRAIN_SUBSET = 6000       # subsample train for speed (raise to 20000 for the final run)
EVAL_N       = 300        # test sequences for CE
BATCH        = 256
LR           = 1e-3
GRAD_CLIP    = 1.0
EPS          = 1e-6       # additive floor in the loss / eval (avoids log 0)


In [ ]:
# ── imports + models ──────────────────────────────────────────────────────────
import os, time, json
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

dev = torch.device(DEVICE)

class ConcatAttentionBlock(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.output_dim = output_dim
        self.W_q = nn.Linear(input_dim, output_dim, bias=False)
        self.W_k = nn.Linear(input_dim, output_dim, bias=False)
        self.W_v = nn.Linear(input_dim, output_dim, bias=False)
        self.W_o = nn.Linear(output_dim, output_dim, bias=False)
    def forward(self, x, mask=None):
        Q, K, Vv = self.W_q(x), self.W_k(x), self.W_v(x)
        sc = torch.bmm(Q, K.transpose(1, 2)) / (self.output_dim ** 0.5)
        if mask is not None: sc = sc + mask
        return torch.cat([x, self.W_o(torch.bmm(F.softmax(sc, -1), Vv))], -1), sc

class DisentangledTransformer(nn.Module):
    def __init__(self, total_vocab, seq_len, attn_dim, bias=True):
        super().__init__()
        self.total_vocab = total_vocab; self.seq_len = seq_len
        self.attn_dim = attn_dim; self.initial_dim = total_vocab + seq_len
        self.layer1 = ConcatAttentionBlock(self.initial_dim, attn_dim)
        self.layer2 = ConcatAttentionBlock(self.initial_dim + attn_dim, attn_dim)
        self.fc_out = nn.Linear(self.initial_dim + 2 * attn_dim, total_vocab, bias=bias)
    def forward(self, x):
        B, T = x.shape
        tok = F.one_hot(x, self.total_vocab).float()
        pos = F.one_hot(torch.arange(T, device=x.device), self.seq_len).float().unsqueeze(0).expand(B, T, -1)
        s = torch.cat([tok, pos], -1)
        m = torch.triu(torch.ones(T, T, device=x.device) * float('-inf'), 1)
        s, _ = self.layer1(s, m); s, _ = self.layer2(s, m)
        return self.fc_out(s)


In [ ]:
# ── W* construction (PerfectInductionTransformer) — reference + warm-start source ─
class _CA(nn.Module):
    def __init__(self, i, o):
        super().__init__(); self.output_dim = o
        self.W_q = nn.Linear(i, o, bias=False); self.W_k = nn.Linear(i, o, bias=False)
        self.W_v = nn.Linear(i, o, bias=False); self.W_o = nn.Linear(o, o, bias=False)
    def forward(self, x, mask=None):
        Q, K, Vv = self.W_q(x), self.W_k(x), self.W_v(x)
        s = torch.bmm(Q, K.transpose(1, 2)) / (self.output_dim ** 0.5)
        if mask is not None: s = s + mask
        return torch.cat([x, self.W_o(torch.bmm(F.softmax(s, -1), Vv))], -1)

class PerfectInductionTransformer(nn.Module):
    def __init__(self, total_vocab, seq_len, vocab_set1):
        super().__init__()
        self.total_vocab = total_vocab; self.seq_len = seq_len; self.vocab_set1 = vocab_set1
        self.initial_dim = total_vocab + seq_len; self.attn_dim = self.initial_dim
        self.layer1 = _CA(self.initial_dim, self.attn_dim)
        self.layer2 = _CA(self.initial_dim + self.attn_dim, self.attn_dim)
        self.fc_out = nn.Linear(self.initial_dim + 2 * self.attn_dim, total_vocab, bias=False)
        self._set_weights()
    def _set_weights(self):
        Vv = self.total_vocab; T = self.seq_len; D = self.attn_dim; V_r = self.vocab_set1
        self.layer1.W_q.weight.data = torch.eye(D) * (D ** 0.5)
        T1 = torch.zeros(D, D)
        for k in range(V_r, Vv): T1[k, :] = -1e9
        for tq in range(T):
            for tk in range(T): T1[Vv + tk, Vv + tq] = -1e9 if tq == tk else -20.0 * (tq - tk)
        self.layer1.W_k.weight.data = T1.t()
        Wv1 = torch.zeros(D, D); Wv1[:Vv, :Vv] = torch.eye(Vv); self.layer1.W_v.weight.data = Wv1
        self.layer1.W_o.weight.data = torch.eye(D)
        in2 = D + D; Wq2 = torch.zeros(D, in2)
        Wq2[:V_r, :V_r] = torch.eye(V_r) * (D ** 0.5) * 1000.0
        Wq2[:V_r, D:D + V_r] = torch.eye(V_r) * (D ** 0.5) * 50.0
        for v in range(Vv): Wq2[V_r, v] = (D ** 0.5) * 10.0
        self.layer2.W_q.weight.data = Wq2
        Wk2 = torch.zeros(D, in2); Wk2[:Vv, D:D + Vv] = torch.eye(Vv)
        for v in range(V_r, Vv): Wk2[V_r, v] = -50000.0
        self.layer2.W_k.weight.data = Wk2
        Wv2 = torch.zeros(D, in2); Wv2[:Vv, :Vv] = torch.eye(Vv); self.layer2.W_v.weight.data = Wv2
        self.layer2.W_o.weight.data = torch.eye(D)
        Wfc = torch.zeros(Vv, in2 + D); Wfc[:Vv, in2:in2 + Vv] = torch.eye(Vv); self.fc_out.weight.data = Wfc
    def forward(self, x):
        B, T = x.shape
        tok = F.one_hot(x, self.total_vocab).float()
        pos = F.one_hot(torch.arange(T, device=x.device), self.seq_len).float().unsqueeze(0).expand(B, T, -1)
        s = torch.cat([tok, pos], -1)
        m = torch.triu(torch.ones(T, T, device=x.device) * float('-inf'), 1)
        s = self.layer1(s, m); s = self.layer2(s, m)
        return self.fc_out(s)

WSTAR = PerfectInductionTransformer(V, SEQ_LEN, V_SIG).to(dev).eval()
print('W* construction built (attn_dim =', WSTAR.attn_dim, ')')


In [ ]:
# ── freeze readout + build model under each init scheme ───────────────────────
def freeze_readout(model, Vv):
    """Pin layer-2 value/output and fc_out to the W* selection -> output is a convex
    combination of input token one-hots (>=0, sums to 1).  Freeze them."""
    ad, idim, d = model.attn_dim, model.initial_dim, model.fc_out.weight.device
    with torch.no_grad():
        model.layer2.W_v.weight.zero_(); model.layer2.W_v.weight[:Vv, :Vv] = torch.eye(Vv, device=d)
        model.layer2.W_o.weight.copy_(torch.eye(ad, device=d))
        model.fc_out.weight.zero_()
        off = idim + ad; model.fc_out.weight[:Vv, off:off + Vv] = torch.eye(Vv, device=d)
        if model.fc_out.bias is not None: model.fc_out.bias.zero_()
    for mod in (model.layer2.W_v, model.layer2.W_o, model.fc_out):
        for p in mod.parameters(): p.requires_grad_(False)

_TRAINABLE = lambda m: [m.layer1.W_q, m.layer1.W_k, m.layer1.W_v, m.layer1.W_o,
                        m.layer2.W_q, m.layer2.W_k]

def build_model(scheme):
    seed = {'default-0': 0, 'default-1': 1, 'default-2': 2}.get(scheme, 0)
    torch.manual_seed(seed)
    m = DisentangledTransformer(V, SEQ_LEN, ATTN_DIM, bias=True).to(dev)
    if scheme == 'small':
        for mod in _TRAINABLE(m): mod.weight.data.mul_(0.1)
    elif scheme == 'large':
        for mod in _TRAINABLE(m): mod.weight.data.mul_(3.0)
    elif scheme == 'wstar':
        m.layer1.W_q.weight.data.copy_(WSTAR.layer1.W_q.weight.data)
        m.layer1.W_k.weight.data.copy_(WSTAR.layer1.W_k.weight.data)
        m.layer1.W_v.weight.data.copy_(WSTAR.layer1.W_v.weight.data)
        m.layer1.W_o.weight.data.copy_(WSTAR.layer1.W_o.weight.data)
        m.layer2.W_q.weight.data.copy_(WSTAR.layer2.W_q.weight.data)
        m.layer2.W_k.weight.data.copy_(WSTAR.layer2.W_k.weight.data)
    freeze_readout(m, V)                      # readout pinned to W* for ALL schemes
    return m


In [ ]:
# ── data + baselines + frozen-readout CE ──────────────────────────────────────
def load_data(split, thr, n_cap):
    b = torch.load(os.path.join(DATA_DIR, split, f'data_x_{thr}.pt'), weights_only=False)
    n = min(n_cap, b['x'].shape[0])
    b['x'] = b['x'][:n]; b['y'] = b['y'][:n]
    if 'transitions' in b: b['transitions'] = b['transitions'][:n]
    return b

@torch.no_grad()
def ce_freeze(model, thr, n=EVAL_N):
    b = load_data('test', thr, n); x = b['x']; y = b['y']
    tot, nt = 0.0, 0
    for i in range(0, x.shape[0], 256):
        xb = x[i:i+256].to(dev); yb = y[i:i+256].to(dev)
        out = model(xb)[:, T0:T1, :]; p = (out + EPS) / (1.0 + V * EPS)
        yw = yb[:, T0:T1]
        tot += F.nll_loss(torch.log(p).reshape(-1, V), yw.reshape(-1), reduction='sum').item()
        nt  += yw.numel()
    return tot / nt

def ctx_ce(thr, n=EVAL_N):
    b = load_data('test', thr, n); x_np = b['x'].numpy(); y_np = b['y'].numpy()
    Vs = V_SIG; nn_ = V - Vs; B = x_np.shape[0]
    tc = np.zeros((B, V, Vs)); pend = np.zeros((B, V)); lr = np.full(B, -1, dtype=np.int64)
    def step(c):
        col = x_np[:, c]; isr = col < Vs; ri = np.where(isr)[0]
        if len(ri):
            rt = col[ri]; oh = (np.arange(Vs) == rt[:, None]).astype(float)
            np.add.at(tc, ri, pend[ri, :, None] * oh[:, None, :]); pend[ri, :] = 0.; pend[ri, rt] = 1.; lr[ri] = rt
        ni = np.where(~isr)[0]
        if len(ni): pend[ni, col[ni]] += 1.
    for i in range(T0): step(i)
    eps = 0.01; tot = 0.; nt = 0
    for t in range(T0, T1):
        v = x_np[:, t]; q = np.where(v < Vs, v, np.maximum(lr, 0)); cnt = tc[np.arange(B), q]
        cs = cnt + eps; ts = cs.sum(1, keepdims=True)
        p = np.zeros((B, V)); p[:, :Vs] = cs / ts * thr; p[:, Vs:] = (1 - thr) / nn_; p = np.maximum(p, 1e-12)
        tot += (-np.log(p[np.arange(B), y_np[:, t]])).sum(); nt += B; step(t)
    return tot / nt

def ideal_ce(thr, n=EVAL_N):
    b = load_data('test', thr, n); x_np = b['x'].numpy(); y_np = b['y'].numpy(); tr = b['transitions']
    Vs = V_SIG; nn_ = V - Vs; B = x_np.shape[0]
    ta = np.array([[tr[i][v] for v in range(Vs)] for i in range(B)], dtype=np.int64)
    tot = 0.; nt = 0; ls = np.zeros(B, dtype=np.int64)
    for t in range(T1):
        iss = x_np[:, t] < Vs; ls = np.where(iss, x_np[:, t], ls)
        if t < T0: continue
        s0 = ta[np.arange(B), ls, 0]; s1 = ta[np.arange(B), ls, 1]
        p = np.zeros((B, V)); p[:, Vs:] = (1 - thr) / nn_
        p[np.arange(B), s0] += thr / 2; p[np.arange(B), s1] += thr / 2
        tot += (-np.log(np.maximum(p[np.arange(B), y_np[:, t]], 1e-12))).sum(); nt += B
    return tot / nt


In [ ]:
# ── train one (init, threshold) -> converged test CE + loss history ───────────
def train_one(scheme, thr):
    b = load_data('train', thr, TRAIN_SUBSET)
    loader = DataLoader(TensorDataset(b['x'], b['y']), batch_size=BATCH, shuffle=True)
    m = build_model(scheme)
    opt = torch.optim.Adam([p for p in m.parameters() if p.requires_grad], lr=LR)
    hist = []
    for ep in range(EPOCHS):
        m.train(); ls = 0.0; ns = 0
        for xb, yb in loader:
            xb = xb.to(dev); yb = yb.to(dev)
            opt.zero_grad(set_to_none=True)
            out = m(xb)[:, T0:T1, :]; p = (out + EPS) / (1.0 + V * EPS)
            loss = F.nll_loss(torch.log(p).reshape(-1, V), yb[:, T0:T1].reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_([q for q in m.parameters() if q.requires_grad], GRAD_CLIP)
            opt.step(); ls += loss.item(); ns += 1
        hist.append(ls / ns)
    return ce_freeze(m, thr), hist


In [ ]:
# ── run everything (resumable cache) ──────────────────────────────────────────
results = json.load(open(RESULTS_PATH)) if os.path.exists(RESULTS_PATH) else {}
for k in ('ctx', 'ideal', 'construction'): results.setdefault(k, {})
for s in INITS: results.setdefault(s, {})

def _save(): json.dump(results, open(RESULTS_PATH, 'w'))

# baselines + W* construction reference
for thr in THRESHOLDS:
    tk = str(thr)
    if tk not in results['ctx']:          results['ctx'][tk] = ctx_ce(thr)
    if tk not in results['ideal']:        results['ideal'][tk] = ideal_ce(thr)
    if tk not in results['construction']: results['construction'][tk] = ce_freeze(WSTAR, thr)
    _save()
    print(f"α={thr}: ctx={results['ctx'][tk]:.4f}  ideal={results['ideal'][tk]:.4f}  "
          f"W*-construction={results['construction'][tk]:.4f}", flush=True)

# GD from each init
for scheme in INITS:
    for thr in THRESHOLDS:
        tk = str(thr)
        if tk in results[scheme]:
            print(f'[skip] {scheme} α={thr} (cached CE={results[scheme][tk]["ce"]:.4f})', flush=True); continue
        t0 = time.time(); ce, hist = train_one(scheme, thr)
        results[scheme][tk] = {'ce': ce, 'loss_hist': hist}; _save()
        print(f'{scheme:10s} α={thr}  test CE={ce:.4f}  (ctx={results["ctx"][tk]:.4f}, '
              f'final_loss={hist[-1]:.4f})  {time.time()-t0:.0f}s', flush=True)
print('\nALL DONE.')


In [ ]:
# ── PLOT: every init vs ctx-count / ideal / W* construction ───────────────────
thrs = THRESHOLDS
def yv(d):  return [d[str(t)] for t in thrs]
def yi(s):  return [results[s][str(t)]['ce'] for t in thrs]

plt.figure(figsize=(11, 6.5))
plt.plot(thrs, yv(results['ideal']), 'k-v', lw=2.5, ms=9, label='ideal-α (Bayes floor)')
plt.plot(thrs, yv(results['ctx']),   '-^', color='firebrick', lw=2.5, ms=9, label='ctx-count')
plt.plot(thrs, yv(results['construction']), '-D', color='green', lw=2.5, ms=9,
         label='W* construction (optimal, frozen readout)')

cmap = plt.cm.plasma(np.linspace(0.0, 0.82, len(INITS)))
for c, scheme in zip(cmap, INITS):
    xs = [t for t in thrs if str(t) in results[scheme]]
    if not xs: continue
    ys = [results[scheme][str(t)]['ce'] for t in xs]
    style = '--*' if scheme == 'wstar' else '-o'
    lw    = 2.4 if scheme == 'wstar' else 1.7
    plt.plot(xs, ys, style, color=c, lw=lw, ms=8, alpha=0.9, label=f'GD init={scheme}')

plt.xlabel('threshold α', fontsize=12)
plt.ylabel(f'test cross-entropy (tokens {T0}-{T1})', fontsize=11)
plt.title('GD (many inits, frozen W* readout) vs ctx-count / ideal / W* construction', fontsize=12)
plt.legend(fontsize=8.5, ncol=2); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'multi_init_ce.png'), dpi=150)
plt.show()
print('Saved -> models/multi_init_ce.png')

# numeric summary: every init's gap above ctx-count
print(f"\n{'init':12s} " + ''.join(f'α={t:<6}' for t in thrs) + '  mean(gap-to-ctx)')
for scheme in INITS:
    if not all(str(t) in results[scheme] for t in thrs): continue
    gaps = [results[scheme][str(t)]['ce'] - results['ctx'][str(t)] for t in thrs]
    row  = ''.join(f'{results[scheme][str(t)]["ce"]:<8.3f}' for t in thrs)
    print(f'{scheme:12s} {row}  +{np.mean(gaps):.3f}')


In [ ]:
# ── convergence check: loss vs epoch (shows each init plateaued) ──────────────
thr_show = THRESHOLDS[len(THRESHOLDS)//2]   # middle threshold
plt.figure(figsize=(9, 5))
for scheme in INITS:
    h = results.get(scheme, {}).get(str(thr_show), {}).get('loss_hist')
    if h: plt.plot(range(1, len(h)+1), h, lw=1.8, label=scheme)
plt.axhline(results['ctx'][str(thr_show)], color='firebrick', ls='--', lw=2, label='ctx-count')
plt.axhline(results['construction'][str(thr_show)], color='green', ls=':', lw=2, label='W* construction')
plt.xlabel('epoch'); plt.ylabel('train loss'); plt.title(f'Convergence at α={thr_show}')
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'multi_init_convergence.png'), dpi=150)
plt.show()
print('Saved -> models/multi_init_convergence.png')
